# Decision Graph — Reaction Flow Demo (Phase 0 + reactive ReAct loop)

This notebook simulates a realistic design session **without** running the live LangGraph agent.
It shows exactly what the decision graph looks like and what the API returns (`{nodes, edges, head}`).

It has two parts:
1. **Phase 0 planning flow** (below) — comprehension `brief` nodes driving a two-turn session with Pareto branches and a backtrack.
2. **The reactive ReAct loop** (last section) — the `thought → act → validate → (debug → retry) → place` self-correction cycle, with the new `thought` / `action ×N` / `validate` / `retry` node types.

## What changed with Phase 0 (BACKEND_PLAN.md)

The agent used to jump straight from the user's message into `read_site`, re-parsing the raw
prompt with keyword regex at every step. **Phase 0 inserts a comprehension layer:** every turn now
begins by extracting a typed **`DesignBrief`** (`START → extract_brief → planner`) and building a
canonical **`SiteModel`**. The downstream chain is now *explained by what the agent understood*,
not by re-reading the text — so the graph reads as a smarter, more logical sequence.

The graph gains a new first-class **`brief`** node (indigo diamond) between every user
**intent** and the first **action**. Its payload is the real `DesignBrief.to_state()` produced by
`agent.brief.extract_brief_fallback` — the same structure the live agent reasons over.

## What changed with the reactive loop

The agent is no longer a one-shot generator. Every geometry it produces is now **validated** (a
`validate_design` checker + LLM brief-judge) and, on failure, the agent **debugs itself** —
diagnoses why, issues a corrective directive, and regenerates, bounded by `max_debug_attempts`.
The last section of this notebook visualizes that loop.

## Simulated session

| Turn | User message | Agent reaction flow |
|------|-------------|---------------------|
| 1 | Place an L-shaped building on the south side | **extract_brief** (1× L building) → read_site (+SiteModel) → generate_shape (L, from brief) → optimize_view (3 Pareto options) → user picks option 2 |
| 2 | Add a T-shaped building near north | **extract_brief** (1× T building) → analyze_remaining → generate_shape (T, from brief) → optimize_two_building (4 Pareto options) → user picks option 1 |
| 3 (backtrack) | Actually I prefer option 3, more view for building 2 | user re-selects option 3 from turn 2's Pareto |

The graph will show the **brief comprehension node** driving each turn, **linear sequences** for
action chains, **branches** for Pareto alternatives, plus the **backtrack fork** when the user
changes their mind.</cell id="cell-dg-00">

In [8]:
import sys
from pathlib import Path

# Resolve the team_04 root whether the notebook is launched from the workspace
# root, team_04/, or team_04/test_notebooks/ (matches test_intent_extraction_P0).
workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / 'team_04',
    workspace_root.parent / 'team_04',
)
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists() and (p / 'backend').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04, or team_04/test_notebooks.')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

from collections import defaultdict
import plotly.graph_objects as go

from backend.decision_graph import (
    DecisionGraph,
    make_intent_node, make_brief_node, make_action_node,
    make_branch_nodes, make_state_node,
)
# Phase 0 comprehension layer — the real typed brief + canonical site model.
from agent.brief import extract_brief_fallback
from agent.tools.site_model import build_site_model

print('Team root      :', TEAM_ROOT)
print('DecisionGraph + Phase 0 brief/site-model imported OK')

Team root      : C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio\team_04
DecisionGraph + Phase 0 brief/site-model imported OK


## Build the mock conversation graph (Phase 0 flow)

Each turn now starts with a real `extract_brief` comprehension node. The brief is produced by the
deterministic `extract_brief_fallback` (the same offline path the live agent degrades to), and the
first action (`read_site`) carries a real `SiteModel` summary. Notice how `generate_shape` no longer
needs to re-parse the prompt — it reads the shape straight from the brief.

In [9]:
g = DecisionGraph()

# Shared site — a 100×100 plot whose south side (side 0) fronts a 20 m street.
SITE = [[0, 0, 0], [100, 0, 0], [100, 100, 0], [0, 100, 0], [0, 0, 0]]
site_model = build_site_model(SITE, {'edge_road_widths': {0: 20.0}})
site_summary = (
    f"sides={len(site_model['sides'])}, "
    f"buildable={site_model['setbacks']['buildable_area_sqm']:.0f} m²"
    if site_model.get('available') else 'unavailable'
)
print(f'SiteModel built — {site_summary}')

# =======================================================================
# TURN 1 — Place L-shaped building
# =======================================================================
prompt_1 = 'Place an L-shaped building on the south side of the site'
i1 = make_intent_node(g, prompt_1)

# --- Phase 0: comprehend the prompt into a typed DesignBrief (real extractor) ---
brief_1 = extract_brief_fallback(prompt_1, {})
b_node_1 = make_brief_node(g, brief_1.to_state(), i1)
spec_1 = brief_1.buildings[0]
print(f'Brief 1 — count={brief_1.building_count}, shape={spec_1.shape_preference}, '
      f'view_w={brief_1.view_weight}, source={brief_1.source}')

# read_site now carries the canonical SiteModel summary; later actions read the brief.
a1 = make_action_node(g, 'read_site',
                      f'{{site_model: {site_summary}}}', b_node_1)
a2 = make_action_node(g, 'generate_building_boundary',
                      f'{{building_type: {spec_1.shape_preference} (from brief), '
                      f'area: 675, requested_position: [50,15]}}', a1)
a3 = make_action_node(g, 'optimize_view_placement',
                      f'{{building_type: {spec_1.shape_preference}, attractors: [south_street], '
                      f'view_weight: {brief_1.view_weight}, pop_size: 50}}', a2)

pareto_1 = [
    {'rank': 1, 'option_id': 't1_opt1', 'combined_score': 0.820,
     'unblocked_view_score': 0.85, 'attractor_view_score': 0.75,
     'rotation_degrees': 0,  'centroid_xy': [28, 18], 'boundary': [], 'outside_area_sqm': 0},
    {'rank': 2, 'option_id': 't1_opt2', 'combined_score': 0.791,
     'unblocked_view_score': 0.80, 'attractor_view_score': 0.78,
     'rotation_degrees': 90, 'centroid_xy': [35, 22], 'boundary': [], 'outside_area_sqm': 0},
    {'rank': 3, 'option_id': 't1_opt3', 'combined_score': 0.765,
     'unblocked_view_score': 0.76, 'attractor_view_score': 0.77,
     'rotation_degrees': 45, 'centroid_xy': [22, 25], 'boundary': [], 'outside_area_sqm': 0},
]
b1 = make_branch_nodes(g, a3, pareto_1)

# User picks option 2
opt2_id = [n['node_id'] for n in g.children_of(b1) if 'Option 2' in n['label']][0]
g.select_node(opt2_id)
sel1 = g.add_node('select', 'Selected: Option 2 (rotation=90°)',
                   parent_id=opt2_id,
                   payload={'selected_option_id': 't1_opt2', 'reason': 'Best attractor score'})

print(f'Turn 1 complete — graph has {g.node_count()} nodes, head={g.current_head()[:8]}...')

# =======================================================================
# TURN 2 — Add a T-shaped building
# =======================================================================
prompt_2 = 'Add a T-shaped building near the north edge facing south'
i2 = make_intent_node(g, prompt_2)

# --- Phase 0: comprehend the second request ---
brief_2 = extract_brief_fallback(prompt_2, {})
b_node_2 = make_brief_node(g, brief_2.to_state(), i2)
spec_2 = brief_2.buildings[0]
print(f'Brief 2 — count={brief_2.building_count}, shape={spec_2.shape_preference}, '
      f'source={brief_2.source}')

a4 = make_action_node(g, 'analyze_remaining_positions',
                      '{placed_building_id: bld_001, site_model: ...}', b_node_2)
a5 = make_action_node(g, 'generate_building_boundary',
                      f'{{building_type: {spec_2.shape_preference} (from brief), '
                      f'area: 600, requested_position: [55, 75]}}', a4)
a6 = make_action_node(g, 'optimize_two_building_placement',
                      f'{{btype_1: {spec_1.shape_preference}, btype_2: {spec_2.shape_preference}, '
                      f'min_separation: 6, pop_size: 60}}', a5)

pareto_2 = [
    {'rank': 1, 'option_id': 't2_opt1', 'combined_score': 0.796,
     'building_1': {'combined_score': 0.81, 'unblocked_view_score': 0.83},
     'building_2': {'combined_score': 0.76, 'unblocked_view_score': 0.74},
     'clearance_between_buildings_m': 8.2, 'boundary': []},
    {'rank': 2, 'option_id': 't2_opt2', 'combined_score': 0.783,
     'building_1': {'combined_score': 0.79, 'unblocked_view_score': 0.81},
     'building_2': {'combined_score': 0.78, 'unblocked_view_score': 0.79},
     'clearance_between_buildings_m': 6.5, 'boundary': []},
    {'rank': 3, 'option_id': 't2_opt3', 'combined_score': 0.771,
     'building_1': {'combined_score': 0.71, 'unblocked_view_score': 0.72},
     'building_2': {'combined_score': 0.85, 'unblocked_view_score': 0.88},
     'clearance_between_buildings_m': 9.1, 'boundary': []},
    {'rank': 4, 'option_id': 't2_opt4', 'combined_score': 0.748,
     'building_1': {'combined_score': 0.68, 'unblocked_view_score': 0.71},
     'building_2': {'combined_score': 0.82, 'unblocked_view_score': 0.85},
     'clearance_between_buildings_m': 11.3, 'boundary': []},
]
b2 = make_branch_nodes(g, a6, pareto_2)

# User picks option 1
opt1_t2_id = [n['node_id'] for n in g.children_of(b2) if 'Option 1' in n['label']][0]
g.select_node(opt1_t2_id)
sel2 = g.add_node('select', 'Selected: Option 1 (best avg score)',
                   parent_id=opt1_t2_id,
                   payload={'selected_option_id': 't2_opt1', 'reason': 'Best combined average'})

print(f'Turn 2 complete — graph has {g.node_count()} nodes')

# =======================================================================
# TURN 3 — Backtrack to option 3
# =======================================================================
# User changes mind — prefers option 3 (better B2 view even if B1 worse)
opt3_t2_id = [n['node_id'] for n in g.children_of(b2) if 'Option 3' in n['label']][0]
g.select_node(opt3_t2_id)   # deselects option 1, selects option 3

sel3 = g.add_node('select', 'Re-selected: Option 3 (B2 view priority)',
                   parent_id=opt3_t2_id,
                   payload={
                       'selected_option_id': 't2_opt3',
                       'reason': 'Higher view score for Building 2 (0.88 vs 0.74)',
                       'backtrack': True,
                   })

print(f'Turn 3 (backtrack) complete — graph has {g.node_count()} nodes')
print(f'Head: {g.current_head()[:8]}...')

# Confirm branch 2 child states
print('\nBranch 2 children after backtrack:')
for c in g.children_of(b2):
    print(f'  {c["label"]:35s} is_selected={c["is_selected"]}')

SiteModel built — sides=4, buildable=7830 m²
Brief 1 — count=1, shape=L, view_w=0.5, source=fallback
Turn 1 complete — graph has 10 nodes, head=34ce4143...
Brief 2 — count=1, shape=T, source=fallback
Turn 2 complete — graph has 21 nodes
Turn 3 (backtrack) complete — graph has 22 nodes
Head: 7e10cff1...

Branch 2 children after backtrack:
  Option 1 (score=0.796)              is_selected=False
  Option 2 (score=0.783)              is_selected=False
  Option 3 (score=0.771)              is_selected=True
  Option 4 (score=0.748)              is_selected=False


## The comprehension layer — what each `brief` node captured

The `brief` nodes are the Phase 0 difference. Each one holds the real `DesignBrief.to_state()`
the agent extracted from the user's message — this is the structure the planner and the
shape-generation repair layer read instead of re-parsing raw text at every step.

In [10]:
import json

brief_nodes = [n for n in g.to_dict()['nodes'] if n['type'] == 'brief']
print(f'{len(brief_nodes)} brief node(s) in the graph:\n')
for n in brief_nodes:
    brief = n['payload']['design_brief']
    shapes = ', '.join(b['shape_preference'] for b in brief['buildings'])
    print(f"• {n['label']}")
    print(f"    shapes        : [{shapes}]")
    print(f"    courtyard     : {brief['courtyard_requested']}  parking: {brief['parking_requested']}")
    print(f"    weights       : view={brief['view_weight']} sun={brief['sun_weight']} align={brief['alignment_weight']}")
    print(f"    rotation_deg  : {brief['requested_rotation_deg']}")
    print(f"    ambiguities   : {brief['ambiguities'] or '(none)'}")
    print(f"    source        : {brief['source']}")
    print()

2 brief node(s) in the graph:

• Brief: 1x [L] (fallback)
    shapes        : [L]
    courtyard     : False  parking: False
    weights       : view=0.5 sun=0.5 align=0.5
    rotation_deg  : None
    ambiguities   : (none)
    source        : fallback

• Brief: 1x [T] (fallback)
    shapes        : [T]
    courtyard     : False  parking: False
    weights       : view=0.5 sun=0.5 align=0.5
    rotation_deg  : None
    ambiguities   : (none)
    source        : fallback



## Selected path trace

Walk from head back to root following `parent_id` — this is the "active design history".

In [11]:
def selected_path(graph):
    """Walk from head to root, return ordered list root → head."""
    path = []
    node_id = graph.current_head()
    while node_id:
        node = graph.get_node(node_id)
        if node is None:
            break
        path.append(node)
        node_id = node.get('parent_id')
    return list(reversed(path))

path = selected_path(g)
print(f'Active path ({len(path)} nodes, root → head):\n')
for i, n in enumerate(path):
    TYPE_ICONS = {'intent': 'USER', 'brief': 'BRIEF', 'action': 'TOOL', 'branch': 'FORK',
                  'select': ' OK ', 'state': 'BLDG'}
    icon = TYPE_ICONS.get(n['type'], '    ')
    prefix = '  ' * min(i, 6)
    print(f'{prefix}[{icon}] {n["label"]}')

Active path (16 nodes, root → head):

[USER] User: Place an L-shaped building on the south side of the site
  [BRIEF] Brief: 1x [L] (fallback)
    [TOOL] Tool: read_site
      [TOOL] Tool: generate_building_boundary
        [TOOL] Tool: optimize_view_placement
          [FORK] Pareto alternatives (3 options)
            [BLDG] Option 2 (score=0.791)
            [ OK ] Selected: Option 2 (rotation=90°)
            [USER] User: Add a T-shaped building near the north edge facing south
            [BRIEF] Brief: 1x [T] (fallback)
            [TOOL] Tool: analyze_remaining_positions
            [TOOL] Tool: generate_building_boundary
            [TOOL] Tool: optimize_two_building_placement
            [FORK] Pareto alternatives (4 options)
            [BLDG] Option 3 (score=0.771)
            [ OK ] Re-selected: Option 3 (B2 view priority)


## API output — what React Flow / D3 receives

`GET /sessions/{id}/decisions` returns `{nodes, edges, head}`.
Edges already have `{id, source, target}` — React Flow needs no transformation.

In [12]:
import json
d = g.to_dict()
print(f'nodes: {len(d["nodes"])}')
print(f'edges: {len(d["edges"])}')
print(f'head:  {d["head"][:8]}...')
print()

# Show node type distribution
from collections import Counter
counts = Counter(n['type'] for n in d['nodes'])
for t, c in sorted(counts.items()):
    print(f'  {t:8s}: {c}')

print('\nFirst edge sample:', json.dumps(d['edges'][0], indent=2))

nodes: 22
edges: 21
head:  7e10cff1...

  action  : 6
  branch  : 2
  brief   : 2
  intent  : 2
  select  : 3
  state   : 7

First edge sample: {
  "id": "8109900d-514d-49ec-a388-83a7272a6098-692cc345-59e9-43a5-8ad6-d8303e404311",
  "source": "8109900d-514d-49ec-a388-83a7272a6098",
  "target": "692cc345-59e9-43a5-8ad6-d8303e404311"
}


## Plotly DAG visualisation

Hierarchical layout:
- **Y axis** = depth level (top = root, down = later decisions)
- **X axis** = horizontal spread within each level
- **Thick solid edge** = selected path (active design history)
- **Thin dashed edge** = unselected branch

Node colours:
- Blue = user intent
- **Indigo diamond = Phase 0 design brief (comprehension step)**
- Orange = tool/action
- Purple = Pareto branch point
- Green = selected option
- Teal = building state / option

In [13]:
def compute_dag_layout(graph_dict):
    """Compute (x, y) positions for a DAG using a simple recursive subtree-width algorithm."""
    nodes = {n['node_id']: n for n in graph_dict['nodes']}
    children_map = defaultdict(list)
    for e in graph_dict['edges']:
        children_map[e['source']].append(e['target'])

    # Find roots (no parent)
    all_children = {e['target'] for e in graph_dict['edges']}
    roots = [n['node_id'] for n in graph_dict['nodes'] if n['node_id'] not in all_children]

    pos = {}
    x_counter = [0]          # mutable counter for leaf x positions

    def place(node_id, depth):
        kids = children_map[node_id]
        if not kids:
            x = x_counter[0]
            x_counter[0] += 1
            pos[node_id] = (x, -depth)
            return x
        child_xs = [place(c, depth + 1) for c in kids]
        x = sum(child_xs) / len(child_xs)
        pos[node_id] = (x, -depth)
        return x

    for r in roots:
        place(r, 0)

    return pos


TYPE_COLORS = {
    'intent':  '#2980b9',   # blue
    'brief':   '#5b4b9e',   # indigo — Phase 0 comprehension
    'action':  '#e67e22',   # orange
    'branch':  '#8e44ad',   # purple
    'select':  '#27ae60',   # green
    'state':   '#16a085',   # teal
}
TYPE_SYMBOLS = {
    'intent': 'diamond',
    'brief':  'diamond-wide',
    'action': 'square',
    'branch': 'star',
    'select': 'circle',
    'state':  'hexagon',
}
TYPE_SIZES = {
    'intent': 28, 'brief': 30, 'action': 20, 'branch': 34, 'select': 24, 'state': 22,
}


def build_plotly_dag(graph, title='Decision Graph'):
    graph_dict = graph.to_dict()
    pos = compute_dag_layout(graph_dict)
    nodes_map = {n['node_id']: n for n in graph_dict['nodes']}
    selected_ids = {n['node_id'] for n in graph_dict['nodes'] if n['is_selected']}
    head_id = graph_dict['head']

    # Build selected-path set (walk from head to root)
    active_path_ids = set()
    nid = head_id
    while nid:
        active_path_ids.add(nid)
        nid = nodes_map[nid].get('parent_id')

    traces = []

    # --- Edges ---
    for e in graph_dict['edges']:
        src_id, tgt_id = e['source'], e['target']
        if src_id not in pos or tgt_id not in pos:
            continue
        x0, y0 = pos[src_id]
        x1, y1 = pos[tgt_id]
        on_active = src_id in active_path_ids and tgt_id in active_path_ids
        color = '#1a252f' if on_active else '#bdc3c7'
        width = 3.0      if on_active else 1.0
        dash  = 'solid'  if on_active else 'dot'
        traces.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode='lines',
            line=dict(color=color, width=width, dash=dash),
            hoverinfo='none', showlegend=False,
        ))

    # --- Nodes (one trace per type for legend) ---
    by_type = defaultdict(list)
    for n in graph_dict['nodes']:
        if n['node_id'] in pos:
            by_type[n['type']].append(n)

    for ntype, ns in by_type.items():
        xs = [pos[n['node_id']][0] for n in ns]
        ys = [pos[n['node_id']][1] for n in ns]
        labels = [n['label'][:30] + ('…' if len(n['label']) > 30 else '') for n in ns]
        hover = [
            f"<b>{n['type'].upper()}</b><br>"
            f"{n['label']}<br>"
            f"selected={n['is_selected']}<br>"
            f"id={n['node_id'][:8]}"
            for n in ns
        ]
        color = TYPE_COLORS.get(ntype, '#7f8c8d')
        symbol = TYPE_SYMBOLS.get(ntype, 'circle')
        size = TYPE_SIZES.get(ntype, 22)

        # Ring highlight for head node
        border_colors = [
            '#f39c12' if n['node_id'] == head_id
            else '#fff' if n['is_selected']
            else '#aaa'
            for n in ns
        ]
        border_widths = [4 if n['node_id'] == head_id else 2 for n in ns]

        traces.append(go.Scatter(
            x=xs, y=ys,
            mode='markers+text',
            marker=dict(
                symbol=symbol, size=size, color=color,
                line=dict(color=border_colors, width=border_widths),
                opacity=[1.0 if n['is_selected'] else 0.45 for n in ns],
            ),
            text=labels,
            textposition='bottom center',
            textfont=dict(size=8, color='#2c3e50'),
            hovertext=hover, hoverinfo='text',
            name=ntype.capitalize(),
            legendgroup=ntype,
            showlegend=True,
        ))

    # --- Head annotation ---
    if head_id in pos:
        hx, hy = pos[head_id]
        traces.append(go.Scatter(
            x=[hx], y=[hy + 0.4],
            mode='text',
            text=['▼ HEAD'],
            textfont=dict(size=9, color='#f39c12'),
            showlegend=False, hoverinfo='none',
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(text=title, font=dict(size=15)),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor='#f8f9fa',
        paper_bgcolor='white',
        legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center'),
        margin=dict(l=20, r=20, t=60, b=60),
        height=700,
    )
    return fig


fig = build_plotly_dag(g, title='Decision Graph — Phase 0 Flow (brief → site model → optimize) with Backtrack')
fig.show()
print('\nLegend: thick solid edge = active path | dashed = explored but not selected')
print('        indigo diamond = Phase 0 design brief | gold ring = current HEAD')
print('        white border = selected | grey = deselected')


Legend: thick solid edge = active path | dashed = explored but not selected
        indigo diamond = Phase 0 design brief | gold ring = current HEAD
        white border = selected | grey = deselected


---

# The reactive ReAct loop — self-validation & self-debug

The graph above shows the *planning* layer (Phase 0 brief → actions → Pareto fork). This section
shows the **reactive self-correction loop** that now wraps every geometry the agent generates —
the difference between a one-shot generator and a true ReAct agent.

The runtime is no longer "generate once and hope". Each candidate runs through:

```
reason (thought) → act (generate) → observe (validate_design) → verdict (validate)
        │                                                              │
        │                                              PASS ───────────┴──► evaluate → place → state
        │                                              FAIL
        └──────────────── debug (diagnose + corrective directive) ◄────────┘   (bounded by max_debug_attempts)
```

The agent **verifies its own output** with `validate_design` (valid polygon, fits the site, no
overlap, area within tolerance) plus an optional LLM brief-judge, and when it fails it **debugs
itself**: it diagnoses *why*, writes a corrective directive, and regenerates with a perturbed seed.
Placement is gated on a passing verdict.

Three new decision-graph node types make this observable — they map 1:1 to the live SSE trace
(`thought` / `validation` / `retry` events):

| Node type | Icon | What it shows |
|-----------|------|---------------|
| `thought` | 🧠 dark diamond | the supervisor's reasoning for the step it is about to take |
| `action`  | 🔧 square (`×N`) | a tool firing, with the **running count** of how many times that tool was used |
| `validate`| ⬡ octagon | the agent's verdict on its own geometry (PASS / FAIL + failing checks) |
| `retry`   | ◀ triangle | a self-debug attempt: diagnosis + corrective directive before regenerating |

The scenario below is the **bounded retry loop in action**: a 1200 m² target where the first
candidate comes out too large, the agent diagnoses the area error, regenerates (note `×2` on the
tools), and the second candidate validates — then places.

In [ ]:
# Build a single-turn graph that depicts the reactive self-correction loop.
from backend.decision_graph import (
    make_thought_node, make_validate_node, make_retry_node,
)

rg = DecisionGraph()

TARGET_AREA = 1200.0

# --- intent + comprehension ------------------------------------------------------
prompt_r = f'Place a {TARGET_AREA:.0f} m² L-shaped building that fits inside the site'
ri = make_intent_node(rg, prompt_r)
brief_r = extract_brief_fallback(prompt_r, {})
rb = make_brief_node(rg, brief_r.to_state(), ri)

# --- Reason -> Act -> Observe (attempt 1) ---------------------------------------
t1 = make_thought_node(rg, 'generate_shape',
                       'Generate an L footprint near the site centroid, ~1200 m².', rb)
gen1 = make_action_node(rg, 'generate_building_boundary',
                        '{building_type: L, area: 1200}', t1,
                        call_count=1, result_summary='geometry_id=geo-1, boundary=8 pts', ok=True)
val_call1 = make_action_node(rg, 'validate_design',
                             '{target_area_sqm: 1200}', gen1,
                             call_count=1, result_summary='validation failed (area)', ok=False)

# --- Verdict: FAIL (footprint too large) ----------------------------------------
v1 = make_validate_node(rg, {
    'passed': False,
    'failures': ['area'],
    'summary': 'Footprint 1452 m² vs target 1200 m² (21% error, tol 25%... near edge).',
    'metrics': {'building_area_sqm': 1452.0, 'target_area_sqm': 1200.0, 'area_error_ratio': 0.21},
}, val_call1)

# --- Self-debug: diagnose + corrective directive --------------------------------
r1 = make_retry_node(rg, 1,
                     'Reduce the footprint area toward 1200 m² and regenerate.', v1,
                     diagnosis='Footprint is ~21% over the requested area.',
                     failures=['area'])

# --- Reason -> Act -> Observe (attempt 2, note the running ×2 counts) ------------
t2 = make_thought_node(rg, 'generate_shape',
                       'Regenerate a smaller L per the debug directive (perturbed seed).', r1)
gen2 = make_action_node(rg, 'generate_building_boundary',
                        '{building_type: L, area: 1200, random_seed: 1009}', t2,
                        call_count=2, result_summary='geometry_id=geo-2, boundary=8 pts', ok=True)
val_call2 = make_action_node(rg, 'validate_design',
                             '{target_area_sqm: 1200}', gen2,
                             call_count=2, result_summary='validation passed', ok=True)

# --- Verdict: PASS --------------------------------------------------------------
v2 = make_validate_node(rg, {
    'passed': True,
    'failures': [],
    'summary': 'All design checks passed (area 1180 m², fits site, no overlap).',
    'metrics': {'building_area_sqm': 1180.0, 'target_area_sqm': 1200.0, 'area_error_ratio': 0.017},
}, val_call2)

# --- Place the validated geometry ------------------------------------------------
t3 = make_thought_node(rg, 'place_building', 'Validation passed — import the footprint.', v2)
place = make_action_node(rg, 'import_building_boundary',
                         '{geometry_id: geo-2}', t3,
                         call_count=1, result_summary='placed', ok=True)
st = make_state_node(rg, place, [{'label': 'Building 1', 'boundary': []}])

# Running per-tool tally (what the UI's "tool usage" chips show) -------------------
from collections import Counter
tool_calls = [n['payload'].get('tool_name') for n in rg.to_dict()['nodes'] if n['type'] == 'action']
print('Reactive loop built —', rg.node_count(), 'nodes')
print('Tool-call counts   :', dict(Counter(tool_calls)))
print('Self-debug attempts:', sum(1 for n in rg.to_dict()['nodes'] if n['type'] == 'retry'))
print('Final verdict       :', 'PASS' if rg.get_node(v2)['payload']['passed'] else 'FAIL')
print('\nReACT trace (root → head):')
for n in rg.to_dict()['nodes']:
    print(f"  {n['type']:9s} | {n['label']}")

In [ ]:
# Teach the existing Plotly renderer about the three reactive-loop node types,
# then draw the loop. build_plotly_dag() reads these maps as globals at call time.
TYPE_COLORS.update({
    'thought':  '#34495e',   # dark slate — the Reason step
    'validate': '#16a085',   # teal — the self-verification verdict
    'retry':    '#d35400',   # burnt orange — a self-debug attempt
})
TYPE_SYMBOLS.update({
    'thought':  'diamond-tall',
    'validate': 'octagon',
    'retry':    'triangle-left',   # points "back" — regenerate
})
TYPE_SIZES.update({
    'thought': 22, 'validate': 26, 'retry': 24,
})

fig_react = build_plotly_dag(
    rg,
    title='Reactive ReAct loop — Reason → Act → Validate → (Self-debug → Retry) → Place',
)
fig_react.show()
print('Legend: 🧠 thought (reason) · 🔧 action ×N (tool + run count) · ⬡ validate (verdict) · ◀ retry (self-debug)')
print('        The FAIL verdict forks into a retry, which regenerates — note the ×2 tool counts on attempt 2.')

## Serialisation round-trip

The graph must survive JSON round-trip (for future Redis/DB storage).

In [14]:
d = g.to_dict()
g2 = DecisionGraph.from_dict(d)
d2 = g2.to_dict()

assert g2.node_count() == g.node_count(), 'node count mismatch after round-trip'
assert d2['head'] == d['head'],           'head mismatch after round-trip'
assert len(d2['edges']) == len(d['edges']), 'edge count mismatch'
print(f'Round-trip OK — {g2.node_count()} nodes, {len(d2["edges"])} edges')
print(f'head: {d2["head"][:8]}...')

Round-trip OK — 22 nodes, 21 edges
head: 7e10cff1...


## Step-by-step replay animation

Simulates how the graph grows node-by-node as events stream in,  
rendered as a **Plotly slider** (one frame per node added).

In [15]:
# Replay: build sub-graphs of size 1..N and snapshot each
all_nodes = g.to_dict()['nodes']  # ordered by insertion
snapshots = []
for k in range(1, len(all_nodes) + 1):
    subset = all_nodes[:k]
    subset_ids = {n['node_id'] for n in subset}
    edges = [e for e in g.to_dict()['edges']
             if e['source'] in subset_ids and e['target'] in subset_ids]
    snapshots.append({'nodes': subset, 'edges': edges,
                      'head': subset[-1]['node_id']})

def pos_for_snapshot(snap):
    # Use final layout but only show subset of nodes
    return compute_dag_layout(snap)

# Build frames
frames = []
# Pre-compute final positions so layout doesn't jump
final_pos = compute_dag_layout(g.to_dict())

for snap in snapshots:
    snap_ids = {n['node_id'] for n in snap['nodes']}
    node_x, node_y, node_text, node_color, node_size = [], [], [], [], []
    for n in snap['nodes']:
        if n['node_id'] not in final_pos:
            continue
        x, y = final_pos[n['node_id']]
        node_x.append(x)
        node_y.append(y)
        node_text.append(n['label'][:25])
        node_color.append(TYPE_COLORS.get(n['type'], '#7f8c8d'))
        node_size.append(TYPE_SIZES.get(n['type'], 22))

    edge_x, edge_y = [], []
    for e in snap['edges']:
        if e['source'] in final_pos and e['target'] in final_pos:
            x0, y0 = final_pos[e['source']]
            x1, y1 = final_pos[e['target']]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]

    frames.append(go.Frame(
        data=[
            go.Scatter(x=edge_x, y=edge_y, mode='lines',
                       line=dict(color='#2c3e50', width=2), hoverinfo='none'),
            go.Scatter(x=node_x, y=node_y, mode='markers+text',
                       marker=dict(color=node_color, size=node_size,
                                   line=dict(color='white', width=2)),
                       text=node_text, textposition='bottom center',
                       textfont=dict(size=7)),
        ],
        name=str(len(snap['nodes'])),
    ))

# Initial frame
first = snapshots[0]
n0 = first['nodes'][0]
x0_init, y0_init = final_pos.get(n0['node_id'], (0, 0))
anim_fig = go.Figure(
    data=[
        go.Scatter(x=[], y=[], mode='lines', line=dict(color='#2c3e50', width=2)),
        go.Scatter(x=[x0_init], y=[y0_init], mode='markers+text',
                   marker=dict(color=[TYPE_COLORS.get(n0['type'], '#7f8c8d')],
                               size=[TYPE_SIZES.get(n0['type'], 22)],
                               line=dict(color='white', width=2)),
                   text=[n0['label'][:25]], textposition='bottom center',
                   textfont=dict(size=7)),
    ],
    frames=frames,
)
anim_fig.update_layout(
    title='Decision graph — step-by-step replay',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#f8f9fa', height=600,
    updatemenus=[dict(
        type='buttons', showactive=False, y=0, x=0.5, xanchor='center',
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, dict(frame=dict(duration=400, redraw=True),
                                  fromcurrent=True)]),
            dict(label='Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0), mode='immediate')]),
        ],
    )],
    sliders=[dict(
        steps=[dict(method='animate', args=[[f.name], dict(mode='immediate')],
                    label=f.name) for f in frames],
        x=0.1, len=0.8, y=0,
        currentvalue=dict(prefix='Nodes added: ', visible=True),
    )],
)
anim_fig.show()

## Notes for the UI implementation

The backend returns `{nodes, edges, head}` from `GET /sessions/{id}/decisions`.

### Node types

`intent | brief | clarify | thought | action | validate | retry | branch | select | state`.

- **`brief`** carries the typed `DesignBrief` in `payload.design_brief` (count, per-building shapes,
  courtyard, objective weights, `source = llm|fallback`) — the comprehension step.
- **`thought`** (`payload: {action, reasoning}`) — the supervisor's reasoning for the step (ReAct "Reason").
- **`action`** (`payload: {tool_name, input_preview, call_count, result_summary, ok}`) — a tool firing,
  with the **running count** of how many times that tool was used this turn.
- **`validate`** (`payload: {passed, failures, summary, metrics, judge}`) — the agent's verdict on its
  own geometry; placement is gated on `passed`.
- **`retry`** (`payload: {attempt, directive, diagnosis, failures}`) — a self-debug attempt before
  regenerating.

`type` is an open string on the wire, so an older UI falls back to a generic node for any unknown type.

### React Flow node data mapping

```js
// Convert backend nodes → React Flow nodes
const rfNodes = nodes.map(n => ({
  id: n.node_id,
  type: n.type,           // 'brief' → BriefNode; 'thought'/'validate'/'retry' → BasicNodes
  data: {
    label: n.label,
    payload: n.payload,   // brief: payload.design_brief · validate: payload.passed/failures
    isSelected: n.is_selected,
    isHead: n.node_id === head,
  },
  position: { x: 0, y: 0 }, // let dagre/elkjs compute layout
}))

// Edges — already have source/target
const rfEdges = edges.map(e => ({
  ...e,
  animated: isOnActivePath(e.source, e.target),
  style: { stroke: isOnActivePath(e.source, e.target) ? '#1a252f' : '#bdc3c7' },
}))
```

### User selects a Pareto option

```js
// POST /sessions/{id}/decisions/{node_id}/select
const res = await fetch(`/sessions/${sessionId}/decisions/${nodeId}/select`, {
  method: 'POST',
  body: JSON.stringify({ reason: 'User clicked option in explorer' }),
})
const { select_node, graph_head } = await res.json()
// Re-fetch graph and update React Flow state
```

### SSE events during chat — the live ReAct trace

Besides `decision` (a new graph node), the chat stream emits the reactive-loop activity events so
the UI can show *what the agent is thinking, which tool it used, how many times, and the result*:

```js
es.addEventListener('decision', e => addNodeToGraph(JSON.parse(e.data)))   // {node_id, type, label, parent_id, is_selected}
es.addEventListener('thought',     e => pushTrace(JSON.parse(e.data)))     // {action, reasoning}
es.addEventListener('tool',        e => pushTrace(JSON.parse(e.data)))     // {name, input_preview, count}
es.addEventListener('tool_result', e => pushTrace(JSON.parse(e.data)))     // {name, ok, summary, count}
es.addEventListener('validation',  e => pushTrace(JSON.parse(e.data)))     // {passed, failures, summary, metrics, judge}
es.addEventListener('retry',       e => pushTrace(JSON.parse(e.data)))     // {attempt, directive, diagnosis, failures}
```

These are driven off each LangGraph node's `on_chain_end` (the agent's tools are in-node Python
calls, not LangChain tools, so `on_tool_start`/`on_tool_end` never fire). `frontend/dashboard/AgentDashboard.tsx`
renders them as a live **Agent Activity** timeline with a per-tool usage tally.</cell id="cell-dg-14">